# 第 5 章:FFN 与 Block —— SwiGLU 与残差流

上一章我们实现了注意力(Attention):token 之间互相「看见」对方,交换信息。但光有信息交换还不够 —— 模型还需要**记住**和**变换**这些信息。

这就是 **FeedForward Network (FFN)** 的职责。如果说 Attention 是「token 间的横向通信」,那 FFN 就是「每个 token 自己的纵向思考」。

本章我们从最朴素的 `Linear→ReLU→Linear` 出发,一步步演进到 minimind 使用的 **SwiGLU**(`model/model_minimind.py:136-146`),再扩展到 **MOEFeedForward**(148-176),最后组装成完整的 **MiniMindBlock**(178-194)。

读完本章,你将理解:

- 为什么 FFN 是 Transformer 里参数量的大头(占每层 76%)
- GLU 门控为什么比单纯 ReLU 更强
- SwiGLU 的三个矩阵 `gate_proj / up_proj / down_proj` 如何流动
- MoE 如何用 router 把 token 分发到多个专家
- residual connection 为什么是深度学习的「生命线」

## 5.1 最朴素的 FFN:Linear → ReLU → Linear

先抛开所有花哨的设计,回到 Transformer 原始论文(Attention is All You Need, 2017)里的 FFN。它极其简单:

$$\text{FFN}(x) = W_2 \cdot \text{ReLU}(W_1 \cdot x + b_1) + b_2$$

两个线性层,中间夹一个 ReLU。`W_1` 把维度从 `hidden_size` 升到 `intermediate_size`(通常是 4 倍),`W_2` 再降回来。

**为什么需要 FFN?**

Attention 是线性的加权求和(softmax 权重 × value)。即使堆叠多层,纯线性的操作仍然只能做线性变换。FFN 引入了**非线性**(ReLU),让模型能拟合复杂函数。同时,FFN 的中间层更宽(4d),提供了**记忆容量** —— 模型把知识「存」在宽中间层的权重里。

一句话:**Attention 负责信息路由,FFN 负责知识存储和非线性变换。**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

hidden_size = 768
intermediate_size = 2432  # minimind 的选择,5.3 节解释为什么

# === 最朴素的 FFN(GPT-2 风格)===
class NaiveFFN(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.w1 = nn.Linear(hidden_size, intermediate_size)     # 升维
        self.w2 = nn.Linear(intermediate_size, hidden_size)     # 降维

    def forward(self, x):
        return self.w2(F.relu(self.w1(x)))

ffn = NaiveFFN(hidden_size, intermediate_size)

# 模拟一个 batch: 2 个序列, 每个序列 10 个 token, 每个 token 是 768 维向量
x = torch.randn(2, 10, hidden_size)
y = ffn(x)

print(f"输入 shape:  {x.shape}  (batch, seq_len, hidden_size)")
print(f"中间 shape:  {ffn.w1(x).shape}  (升维到 intermediate_size)")
print(f"输出 shape:  {y.shape}  (降回 hidden_size)")
print()

# 参数量
p1 = hidden_size * intermediate_size + intermediate_size  # w1 + b1
p2 = intermediate_size * hidden_size + hidden_size        # w2 + b2
print(f"NaiveFFN 参数: w1={p1:,} + w2={p2:,} = {p1+p2:,}")
print(f"(注意:minimind 的 Linear 没有 bias,所以实际更少)")

### 维度流动图

```
(b, T, 768) ──w1──→ (b, T, 2432) ──ReLU──→ (b, T, 2432) ──w2──→ (b, T, 768)
                     升维 4×宽              非线性激活             降维回来
```

每个 token **独立**经过这个变换 —— FFN 没有 token 间的交互(那是 Attention 的事)。你可以把 FFN 理解为:对序列里每个位置,套用同一个「两层 MLP」。

&nbsp;

---

## 5.2 门控 FFN:GLU 的引入

朴素 FFN 的问题:`ReLU(W_1 x)` 是一个硬决策 —— 负数直接变 0,信息不可逆地丢失。

**GLU (Gated Linear Unit)** 的思路:与其用激活函数「砍掉」信息,不如用另一个线性变换来「**门控**」—— 让模型自己学习该放行多少:

$$\text{GLU}(x) = \underbrace{(W_1 x)}_{\text{value}} \odot \underbrace{\sigma(W_{\text{gate}} x)}_{\text{gate}}$$

其中 $\sigma$ 是 sigmoid,$\odot$ 是逐元素乘法。

- **value 路径** $W_1 x$:线性变换后的「内容」
- **gate 路径** $\sigma(W_{\text{gate}} x)$:0 到 1 之间的「阀门」,控制每个维度放行多少

**为什么门控更好?**

1. **软性过滤**:gate 是连续的(0.0~1.0),而非 ReLU 的硬截断(0 或原值)
2. **数据驱动**:gate 的值由输入 $x$ 决定,不同输入有不同的过滤模式
3. **梯度更流畅**:sigmoid 处处可导,不像 ReLU 在负区梯度为 0

代价是:多了一个矩阵 $W_{\text{gate}}$,参数量增加 50%。

In [ ]:
# === GLU 风格的 FFN ===
class GLUFFN(nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.w_gate = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.w_value = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.w_down  = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x):
        gate = torch.sigmoid(self.w_gate(x))   # (b, T, d_ff)  值域 [0,1]
        value = self.w_value(x)                 # (b, T, d_ff)  线性内容
        return self.w_down(gate * value)        # 门控后降维

glu = GLUFFN(hidden_size, intermediate_size)
x = torch.randn(2, 10, hidden_size)
y_glu = glu(x)

print(f"GLU FFN 输出 shape: {y_glu.shape}")
print()

# 对比门控效果
gate_vals = torch.sigmoid(glu.w_gate(x))
print(f"gate 值域: min={gate_vals.min():.4f}, max={gate_vals.max():.4f}, mean={gate_vals.mean():.4f}")
print(f"→ gate 不是 0/1 硬截断,而是 [0,1] 的软性过滤")
print()

# 参数量对比
p_naive = 2 * hidden_size * intermediate_size   # w1 + w2
p_glu   = 3 * hidden_size * intermediate_size   # gate + value + down
print(f"Naive FFN (2 矩阵): {p_naive:,}")
print(f"GLU FFN   (3 矩阵): {p_glu:,}  (+50%)")

&nbsp;

---

## 5.3 SwiGLU:SiLU 激活的门控

minimind 用的是 **SwiGLU**(SiLU-Gated Linear Unit),出自 Llama 论文(GLU Variants Improve Transformer, 2020)。它把 GLU 里的 sigmoid 换成了 **SiLU**(也叫 Swish):

$$\text{SwiGLU}(x) = W_{\text{down}} \cdot \big(\text{silu}(W_{\text{gate}} x) \odot W_{\text{up}} x\big)$$

其中 $\text{silu}(x) = x \cdot \sigma(x)$。

对应到 minimind 代码(`model/model_minimind.py:136-146`):

```python
class FeedForward(nn.Module):
    def __init__(self, config):
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.up_proj   = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.act_fn    = ACT2FN['silu']

    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))
```

### 为什么用 SiLU 而不是 ReLU 或 Sigmoid?

| 激活函数 | 公式 | 特点 |
|---|---|---|
| ReLU | $\max(0, x)$ | 硬截断,负区梯度为 0(「死神经元」) |
| Sigmoid | $\sigma(x)$ | 饱和(两端梯度趋于 0),输出非零中心 |
| **SiLU** | $x \cdot \sigma(x)$ | **负区也有微小负值,梯度处处非零,平滑** |

SiLU 的关键优势:**对负输入不是直接归零,而是输出一个小的负值**。这让信息流更平滑,梯度更稳定。Google 和 DeepMind 的研究都发现 SiLU 在深度 Transformer 里一致优于 ReLU。

In [ ]:
# === minimind 的 FeedForward(SwiGLU)===
class FeedForward(nn.Module):
    """model/model_minimind.py:136-146 的精确复现"""
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.up_proj   = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.act_fn    = F.silu  # silu(x) = x * sigmoid(x)

    def forward(self, x):
        # 核心公式:down_proj( silu(gate_proj(x)) * up_proj(x) )
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

ffn_swiglu = FeedForward(hidden_size, intermediate_size)
x = torch.randn(2, 10, hidden_size)
y = ffn_swiglu(x)

# 逐步打印 shape
gate = ffn_swiglu.gate_proj(x)
act  = ffn_swiglu.act_fn(gate)
up   = ffn_swiglu.up_proj(x)
mid  = act * up
out  = ffn_swiglu.down_proj(mid)

print("=== SwiGLU 逐步 shape 流动 ===")
print(f"x          : {x.shape}        ← 输入 (b, T, 768)")
print(f"gate_proj  : {gate.shape}  ← 升维 (b, T, 2432)")
print(f"silu(gate) : {act.shape}  ← 激活,值域变化")
print(f"up_proj    : {up.shape}  ← 升维 (b, T, 2432)")
print(f"silu*up    : {mid.shape}  ← 门控相乘")
print(f"down_proj  : {out.shape}        ← 降维回 (b, T, 768)")
print()

# 验证 SiLU 的特性:负区不归零
demo = torch.linspace(-4, 4, 9)
print(f"x      : {demo.tolist()}")
print(f"ReLU   : {F.relu(demo).tolist()}")
print(f"SiLU   : {[round(v, 4) for v in F.silu(demo).tolist()]}")
print(f"→ SiLU 在负区输出小的负值(如 x=-1 → -0.269),ReLU 直接归零")

### intermediate_size = 2432 的 π 之谜

SwiGLU 有 3 个矩阵(gate/up/down),而朴素 FFN 只有 2 个。为了不让参数量膨胀太多,中间层要**收窄**:

$$3 \times d \times d_{\text{ff}} \approx 4 \times d^2 \quad \Rightarrow \quad d_{\text{ff}} \approx \frac{4}{3}d$$

但 minimind(跟随 Llama/Qwen)用了更优雅的公式:

$$d_{\text{ff}} = \left\lceil \frac{d \times \pi}{64} \right\rceil \times 64$$

代入 $d=768$:$\lceil 768 \times 3.14159 / 64 \rceil \times 64 = \lceil 37.70 \rceil \times 64 = 38 \times 64 = \mathbf{2432}$。

**为什么是 π?** 这其实是个近似:目标比例 $\approx 4/3 \approx 1.33$,而 $\pi/2 \approx 1.57$。Llama 团队发现略宽一点效果更好,而 π 恰好给出了一个「好看又能用」的数字。最后的 `× 64` 是为了对齐 GPU Tensor Core 的矩阵乘法效率(64 的倍数最高效)。

> 这个公式在 `MiniMindConfig.__init__` 的第 25 行:`self.intermediate_size = math.ceil(hidden_size * math.pi / 64) * 64`(第 3 章详解)。

In [ ]:
import math

hidden_size = 768
d_ff = math.ceil(hidden_size * math.pi / 64) * 64

print("=== intermediate_size 计算过程 ===")
print(f"hidden_size × π = {hidden_size} × {math.pi:.6f} = {hidden_size * math.pi:.4f}")
print(f"÷ 64            = {hidden_size * math.pi / 64:.4f}")
print(f"ceil()          = {math.ceil(hidden_size * math.pi / 64)}")
print(f"× 64            = {d_ff}  ← 最终的 intermediate_size")
print()

# 参数量验证
p = 3 * hidden_size * d_ff
print(f"SwiGLU 参数/层 = 3 × {hidden_size} × {d_ff} = {p:,} ({p/1e6:.2f}M)")
print(f"占每层参数的 {p / (p + 1769664 + 1536):.1%}  ← FFN 是参数大头")

&nbsp;

---

## 5.4 激活函数对比:ReLU vs GELU vs SiLU

让我们可视化三种激活函数的差异,理解为什么 SwiGLU 选择 SiLU。

- **ReLU**: $f(x) = \max(0, x)$ — 最古老,简单粗暴
- **GELU**: $f(x) = x \cdot \Phi(x)$,其中 $\Phi$ 是标准正态的 CDF — BERT/GPT-2 使用
- **SiLU**: $f(x) = x \cdot \sigma(x)$ — Llama/Qwen/minimind 使用

GELU 和 SiLU 形态相似(都是「平滑的 ReLU」),但 SiLU 用 sigmoid 近似 CDF,计算更便宜。

In [ ]:
import matplotlib
matplotlib.use('Agg')  # 无显示器环境
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(-4, 4, 200)
relu = np.maximum(0, x)

# GELU: x * Phi(x), 用 erf 近似
from scipy.special import erf
gelu = 0.5 * x * (1 + erf(x / np.sqrt(2)))

# SiLU: x * sigmoid(x)
sigmoid = 1 / (1 + np.exp(-x))
silu = x * sigmoid

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# 左图:三条曲线
ax = axes[0]
ax.plot(x, relu, label='ReLU', linewidth=2)
ax.plot(x, gelu, label='GELU (BERT/GPT-2)', linewidth=2, linestyle='--')
ax.plot(x, silu, label='SiLU (Llama/minimind)', linewidth=2, linestyle='-.')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('Activation Functions')
ax.legend()
ax.set_ylim(-1, 4)
ax.grid(True, alpha=0.3)

# 右图:放大负区
ax2 = axes[1]
mask = x < 0.5
ax2.plot(x[mask], relu[mask], label='ReLU', linewidth=2)
ax2.plot(x[mask], gelu[mask], label='GELU', linewidth=2, linestyle='--')
ax2.plot(x[mask], silu[mask], label='SiLU', linewidth=2, linestyle='-.')
ax2.axhline(0, color='gray', linewidth=0.5)
ax2.set_xlabel('x')
ax2.set_ylabel('f(x)')
ax2.set_title('Negative Region (zoomed)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('activation_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("图表已保存: activation_comparison.png")
print()
print("关键观察:")
print("  • ReLU 在 x<0 时恒为 0(硬截断,梯度消失)")
print("  • GELU 在 x≈-0.17 处有最小值 ≈ -0.17(平滑负值)")
print("  • SiLU 在 x≈-1.28 处有最小值 ≈ -0.28(更大的负值,更强表达)")

> **小结**:SwiGLU = SiLU 激活 + GLU 门控。它用三个矩阵(`gate_proj` / `up_proj` / `down_proj`)换取了比朴素 FFN 更强的表达能力,而 `intermediate_size=2432` 通过 π 公式控制了参数预算。每层 FFN 参数 5.6M,占整层 76% —— FFN 是 minimind 的「记忆仓库」。

&nbsp;

---

## 5.5 MoE FFN:混合专家(bonus)

当 `config.use_moe=True` 时,每层的 `FeedForward` 被替换为 `MOEFeedForward`(`model_minimind.py:148-176`)。核心思想:**用多个 FFN「专家」替代单个 FFN,每次只激活其中一个**。

### 架构

```
                ┌──→ Expert 0 (FFN) ──┐
x ──→ router ──┼──→ Expert 1 (FFN) ──┼──→ 加权汇总 ──→ y
   (gate线性层) ├──→ Expert 2 (FFN) ──┤
                └──→ Expert 3 (FFN) ──┘
```

**router**(门控网络)是一个 `hidden_size → num_experts` 的线性层 + softmax,输出每个 token 对每个专家的偏好概率。然后取 top-1(只选概率最高的 1 个专家)。

### 代码逐步拆解

```python
class MOEFeedForward(nn.Module):
    def __init__(self, config):
        self.gate    = nn.Linear(hidden_size, num_experts, bias=False)  # router
        self.experts = nn.ModuleList([FeedForward(...) for _ in range(num_experts)])

    def forward(self, x):
        scores   = F.softmax(self.gate(x_flat), dim=-1)        # 路由概率
        topk_w, topk_idx = torch.topk(scores, k=1, dim=-1)     # top-1 选择
        topk_w   = topk_w / topk_w.sum()                        # 归一化
        y        = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = (topk_idx == i)                              # 哪些 token 选了专家 i
            y.index_add_(0, token_idx, expert(x) * weight)      # 累加结果
        return y
```

In [ ]:
# === MOEFeedForward 简化复现(聚焦路由逻辑)===
class MOEFeedForward(nn.Module):
    def __init__(self, hidden_size, intermediate_size, num_experts=4, num_experts_per_tok=1):
        super().__init__()
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.gate = nn.Linear(hidden_size, num_experts, bias=False)
        self.experts = nn.ModuleList([
            FeedForward(hidden_size, intermediate_size) for _ in range(num_experts)
        ])

    def forward(self, x):
        b, T, d = x.shape
        x_flat = x.view(-1, d)                              # (b*T, d)

        # 1. Router: 每个 token 对每个专家的概率
        scores = F.softmax(self.gate(x_flat), dim=-1)       # (b*T, num_experts)

        # 2. Top-1 路由: 选概率最高的 1 个专家
        topk_w, topk_idx = torch.topk(scores, k=self.num_experts_per_tok, dim=-1, sorted=False)
        topk_w = topk_w / (topk_w.sum(dim=-1, keepdim=True) + 1e-20)  # 归一化

        # 3. 分发: 每个 token 只经过被选中的专家
        y = torch.zeros_like(x_flat)
        route_count = [0] * self.num_experts
        for i, expert in enumerate(self.experts):
            mask = (topk_idx == i).any(dim=-1)               # 哪些 token 选了专家 i
            if mask.any():
                idx = mask.nonzero().flatten()
                w = topk_w[idx].max(dim=-1)[0].unsqueeze(-1) # 取 top-1 权重
                y.index_add_(0, idx, expert(x_flat[idx]) * w)
                route_count[i] = len(idx)

        # 4. Aux loss(负载均衡, 训练时计算)
        load = torch.zeros(self.num_experts)
        for i in range(self.num_experts):
            load[i] = route_count[i]
        load = load / load.sum()
        prob_mean = scores.mean(0)
        self.aux_loss = (load * prob_mean).sum() * self.num_experts

        return y.view(b, T, d), route_count, self.aux_loss

# 测试
moe = MOEFeedForward(768, 2432, num_experts=4)
x = torch.randn(2, 20, 768)
y, routing, aux = moe(x)

print(f"输入: {x.shape} → 输出: {y.shape}")
print(f"\n=== 路由分布(20 token × 2 seq = 40 token) ===")
for i, count in enumerate(routing):
    print(f"  Expert {i}: 收到 {count} 个 token ({count/40:.0%})")
print(f"\naux_loss = {aux.item():.4f}")
print(f"(aux_loss 越小 = 负载越均衡; 理想情况每个专家收 25%)")

### aux_loss:为什么需要负载均衡?

如果没有约束,router 可能陷入「**赢者通吃**」:一开始某个专家碰巧表现好 → 梯度更新让它更强 → 更多 token 选它 → 其他专家饿死。

`aux_loss`(auxiliary loss)通过惩罚不均衡来打破这个恶性循环:

$$L_{\text{aux}} = \alpha \cdot N \cdot \sum_{i=1}^{N} \bar{f}_i \cdot \bar{P}_i$$

- $\bar{f}_i$:专家 $i$ 实际收到的 token 比例(fraction)
- $\bar{P}_i$:router 给专家 $i$ 的平均概率(probability)
- $N$:专家数;$\alpha = 5 \times 10^{-4}$:很小的系数

当所有专家均匀分担时,$\bar{f}_i = \bar{P}_i = 1/N$,$L_{\text{aux}}$ 达到最小值 1.0。任何不均衡都会让 $L_{\text{aux}}$ 增大,从而通过梯度推动 router 更均衡地分发。

### Dense vs MoE 参数量

| 配置 | 总参数量 | 每层 FFN | 每 token 激活 |
|---|---|---|---|
| Dense (`use_moe=False`) | 64M | 5.6M | 5.6M |
| MoE (`use_moe=True, 4 experts`) | **198M** | **22.4M** | **5.6M** |

MoE 用 3 倍的总参数(4 个专家),但每个 token 只激活 1 个 → **计算成本与 dense 相同**。这就是 MoE 的核心价值:以相同的 FLOPS 换取 3 倍的知识容量。

In [ ]:
# Dense vs MoE 参数量对比
d, d_ff = 768, 2432
num_experts = 4
num_layers = 8

# 每层 FFN 参数
dense_ffn = 3 * d * d_ff                     # 5,603,328
moe_ffn   = num_experts * dense_ffn + d * num_experts  # 4 个专家 + router

print(f"Dense FFN / 层:  {dense_ffn:>12,} ({dense_ffn/1e6:.2f}M)")
print(f"MoE FFN / 层:    {moe_ffn:>12,} ({moe_ffn/1e6:.2f}M)")
print(f"  ├─ {num_experts} 专家: {num_experts * dense_ffn:>12,}")
print(f"  └─ router:      {d * num_experts:>12,}")
print()

# 每 token 激活的 FFN 参数(top-1)
print(f"每 token 激活:    {dense_ffn:>12,} (只过 1 个专家,与 dense 相同!)")
print()

# 全模型粗估(不含 embedding/norm)
dense_total = num_layers * dense_ffn
moe_total   = num_layers * moe_ffn
print(f"8 层 FFN 总计:")
print(f"  Dense: {dense_total/1e6:.1f}M")
print(f"  MoE:   {moe_total/1e6:.1f}M  ({moe_total/dense_total:.1f}× 容量)")

> **小结**:MoE 用 router 把 token 路由到多个专家,以 3 倍参数容量换取相同的计算成本。`aux_loss` 是负载均衡的「保险丝」,防止赢者通吃。第 15 章会详解 MoE 的训练和蒸馏。

&nbsp;

---

## 5.6 残差连接:深度学习的生命线

有了 Attention 和 FFN,现在要把它们组装成 Block。但在这之前,必须先理解**残差连接(residual connection)**。

### 为什么需要残差?

深度网络的致命问题:**梯度消失**。反向传播时,梯度要穿过很多层,每经过一个非线性变换就会衰减。20 层以后,底层的梯度可能已经趋近于 0 —— 网络根本学不动。

残差连接的解法极其简单 —— **把输入加到输出上**:

$$h_{\text{out}} = h + F(h)$$

这样梯度反向传播时,有一条「**高速公路**」可以直接跳过 $F$:

$$\frac{\partial h_{\text{out}}}{\partial h} = 1 + F'(h)$$

那个 `+1` 保证了梯度至少为 1,不会消失。这让堆叠几十上百层成为可能。

### Pre-Norm vs Post-Norm

残差连接放在 LayerNorm 的哪一侧,是一个关键设计选择:

| 方案 | 公式 | 特点 |
|---|---|---|
| **Post-Norm**(原始 Transformer) | $h = \text{Norm}(h + F(h))$ | Norm 在残差之后,深层不稳定 |
| **Pre-Norm**(GPT-2/Llama/minimind) | $h = h + F(\text{Norm}(h))$ | Norm 在残差之内,**更稳定** |

minimind 用 **Pre-Norm**(RMSNorm):先做 LayerNorm,再进 Attention/FFN,最后加回残差流。这让残差流里始终是「干净」的累加,训练更稳定。

> 第 6 章(RMSNorm)会详解为什么用 RMSNorm 而不是 LayerNorm。

In [ ]:
# 演示残差连接对梯度的影响
import torch

# 一个 10 层的模拟网络,每层是一个非线性变换
x = torch.randn(768, requires_grad=True)

# === 无残差: 梯度层层衰减 ===
h = x
for i in range(10):
    h = torch.tanh(h * 0.5)  # 模拟一层变换
h.sum().backward()
grad_no_residual = x.grad.abs().mean()
x.grad = None

# === 有残差: 梯度畅通 ===
h = x
for i in range(10):
    h = h + torch.tanh(h * 0.5)  # 残差连接!
h.sum().backward()
grad_residual = x.grad.abs().mean()

print(f"=== 10 层网络后的输入梯度 ===")
print(f"无残差: {grad_no_residual:.6e}  (梯度严重衰减)")
print(f"有残差: {grad_residual:.6e}  (梯度保持)")
print(f"比值:   {grad_residual / grad_no_residual:.1f}×")
print()
print("→ 残差连接让梯度 '走高速公路',深度网络才能训练")

&nbsp;

---

## 5.7 MiniMindBlock:组装 Attention + FFN

现在把残差、RMSNorm、Attention、FFN 组装成一个完整的 Transformer Block。

对应代码(`model_minimind.py:178-194`):

```python
class MiniMindBlock(nn.Module):
    def __init__(self, layer_id, config):
        self.self_attn           = Attention(config)
        self.input_layernorm     = RMSNorm(config.hidden_size)
        self.post_attention_layernorm = RMSNorm(config.hidden_size)
        self.mlp = FeedForward(config) if not config.use_moe else MOEFeedForward(config)

    def forward(self, hidden_states, ...):
        # Pre-Norm Attention 子层
        residual = hidden_states
        hidden_states = self.self_attn(self.input_layernorm(hidden_states), ...)
        hidden_states = residual + hidden_states          # 残差 1

        # Pre-Norm FFN 子层
        hidden_states = hidden_states + self.mlp(self.post_attention_layernorm(hidden_states))
        return hidden_states                              # 残差 2
```

### 数据流

```
h ──┬──→ input_ln ──→ Attention ──→ + ──┬──→ post_attn_ln ──→ FFN ──→ + ──→ h_out
    └────────── residual 1 ─────────────┘└────────── residual 2 ─────────┘
```

注意两个关键点:

1. **两个残差连接**:一个包住 Attention,一个包住 FFN。残差流 `h` 贯穿整个 Block。
2. **两个 LayerNorm**:分别用在 Attention 前和 FFN 前。它们是独立的(不共享权重)。
3. **MoE 开关**:如果 `use_moe=True`,`self.mlp` 自动切换为 `MOEFeedForward`,其余结构不变。这是 minimind 的优雅设计 —— Dense 和 MoE 共享同一个 Block 骨架。

In [ ]:
# === MiniMindBlock 简化复现(聚焦残差结构)===
class RMSNorm(nn.Module):
    """简化版 RMSNorm"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

class SimpleAttention(nn.Module):
    """占位用的简化 Attention(真正的 Attention 在第 4 章)"""
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
    def forward(self, x):
        return self.proj(x)

class MiniMindBlock(nn.Module):
    """model_minimind.py:178-194 的结构复现"""
    def __init__(self, dim, use_moe=False):
        super().__init__()
        self.self_attn = SimpleAttention(dim)
        self.input_layernorm = RMSNorm(dim)
        self.post_attention_layernorm = RMSNorm(dim)
        self.mlp = FeedForward(dim, 2432)  # 或 MOEFeedForward

    def forward(self, h):
        # 子层 1: Pre-Norm Attention + 残差
        residual = h
        h = self.self_attn(self.input_layernorm(h))
        h = residual + h                         # 残差连接 1

        # 子层 2: Pre-Norm FFN + 残差
        h = h + self.mlp(self.post_attention_layernorm(h))  # 残差连接 2
        return h

# 测试: 单个 Block
block = MiniMindBlock(768)
x = torch.randn(2, 10, 768)
y = block(x)
print(f"Block 输入: {x.shape}")
print(f"Block 输出: {y.shape}  (维度不变,这是残差流的特征)")
print()

# 堆叠 8 个 Block(模拟完整模型)
blocks = nn.ModuleList([MiniMindBlock(768) for _ in range(8)])
h = x
print("=== 8 层 Block 堆叠 ===")
for i, blk in enumerate(blocks):
    h_norm = h.std().item()
    h = blk(h)
    print(f"  Layer {i}: h.std() = {h_norm:.4f} → {h.std().item():.4f}")
print(f"\n最终输出: {h.shape}")
print(f"→ 残差流让 hidden 始终保持 768 维,8 层后仍然稳定")

### Pre-Norm 的稳定性

注意上面 8 层堆叠后,`h.std()` 始终稳定 —— 这正是 Pre-Norm 的优势。如果用 Post-Norm(`Norm(h + F(h))`),深层容易数值爆炸。

minimind 的 8 层不算深,但这种 Pre-Norm 设计让它可以平滑扩展到更深的配置(只需改 `num_hidden_layers`)。这也是 GPT-2 之后几乎所有现代 LLM 的标准做法。

&nbsp;

---

## Summary and takeaways

本章从朴素 FFN 演进到 SwiGLU,再扩展到 MoE,最后组装成 Block:

| 组件 | 核心公式 | 参数占比 |
|---|---|---|
| **Naive FFN** | $W_2 \cdot \text{ReLU}(W_1 x)$ | — |
| **SwiGLU FFN** | $W_{\text{down}}(\text{silu}(W_{\text{gate}} x) \odot W_{\text{up}} x)$ | **每层 76%** |
| **MoE FFN** | router 选 top-1 专家 + aux_loss 均衡 | 3× 总参数,1× 激活 |
| **Block** | $h = h + \text{Attn}(\text{LN}(h))$;<br>$h = h + \text{FFN}(\text{LN}(h))$ | — |

**关键认知**:

1. **FFN 是知识仓库**:Attention 负责 token 间路由,FFN 负责存储和非线性变换。SwiGLU 的 3 个矩阵占了每层 76% 的参数。
2. **门控 > 硬截断**:GLU 用学习到的 gate 做软性过滤,比 ReLU 的硬截断更优雅。SiLU 激活让负区也有微小梯度。
3. **intermediate_size = ⌈768·π/64⌉·64 = 2432**:π 公式在参数预算和表达力之间取平衡,`×64` 对齐 Tensor Core。
4. **MoE = 稀疏激活**:4 个专家让总参数膨胀 3 倍,但每 token 只激活 1 个 → 相同 FLOPS、3 倍容量。`aux_loss` 防止赢者通吃。
5. **残差连接是生命线**:`h + F(h)` 给梯度一条高速公路,让深层网络可训练。Pre-Norm 比 Post-Norm 更稳定。

> 下一章我们会深入 **RMSNorm** —— 比 LayerNorm 更省算力的归一化方案,以及它为什么和 RoPE 配合得最好。

- 精简复习版见 [`./ffn.ipynb`](./ffn.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)

下一章:[第 6 章 · RMSNorm 与 LayerNorm](../ch06/01_main-chapter-code/README.md)